In [0]:
books_df = spark.table("workspace.default.books")

In [0]:
#parse genres column into array column
from pyspark.sql import functions as F

dim_books_df = books_df.withColumn("genres_array", F.split(F.regexp_replace(F.col("genres"), r"[\[\]']", ""), r",\s*"))

In [0]:
dim_books_df = dim_books_df.withColumn("authors_array", F.split(F.regexp_replace(F.col("authors"), r"[\[\]']", ""),r",\s*"))

In [0]:
#cast numeric columns to proper numbers
dim_books_df = dim_books_df.withColumn("original_publication_year", F.col("original_publication_year").cast("int")).withColumn("pages", F.col("pages").cast("int"))

In [0]:
from pyspark.sql.window import Window

#ntile splits all books into 4 equal size groups based on rating_count

window_spec = Window.orderBy(F.col("ratings_count"))
dim_books_df = dim_books_df.withColumn("popularity_bucket", F.ntile(4).over(window_spec))

In [0]:
dim_books_df = dim_books_df.select("book_id", "title",
    F.col("authors_array").alias("authors"),
    F.col("genres_array").alias("genres"),
    "original_publication_year", "pages","average_rating", "ratings_count", "popularity_bucket")

In [0]:
def quality_gate(df, table_name, key_column, min_expected_rows=1):
    row_count = df.count()
    null_keys = df.filter(df[key_column].isNull()).count()
    duplicate_keys = row_count - df.dropDuplicates([key_column]).count()
    if row_count < min_expected_rows:
        raise ValueError(f"[{table_name}] Row count {row_count} below minimum {min_expected_rows} — aborting.")
    if null_keys > 0:
        raise ValueError(f"[{table_name}] Found {null_keys} null values in key column '{key_column}' — aborting.")
    if duplicate_keys > 0:
        df = df.dropDuplicates([key_column])
    return df

dim_books_df = quality_gate(dim_books_df, "dim_books", key_column="book_id", min_expected_rows=9000)

In [0]:
from delta.tables import DeltaTable

def upsert_delta(spark, source_df, target_table_name, merge_key):
    if not spark.catalog.tableExists(target_table_name):
        source_df.write.format("delta").saveAsTable(target_table_name)
        return
    target = DeltaTable.forName(spark, target_table_name)
    (target.alias("t").merge(source_df.alias("s"), f"t.{merge_key} = s.{merge_key}")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

upsert_delta(spark, dim_books_df, "workspace.default.dim_books", merge_key="book_id")

In [0]:
display(spark.table("workspace.default.dim_books").limit(5))